# Stage 2 Data Analysis Athena

### Imports

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory

In [7]:
# =============================================================================
# Compare two settings files written by dump_sim_settings.py
#
# Runs anywhere with plain Python (notebook, terminal), no Isaac Sim needed.
# Prints only what differs: entries missing in one scene and different values.
# =============================================================================

import json

FILE_A = "/home/tommy/Documents/autonomous-systems/athena/athena-data-analysis/sim-to-real-gap/stage-2-testing/compare-stage1-stage2/scene1_settings.json"   # reference (scene 1)
FILE_B = "/home/tommy/Documents/autonomous-systems/athena/athena-data-analysis/sim-to-real-gap/stage-2-testing/compare-stage1-stage2/scene1_settings-new.json"   # scene to check (scene 2)
TOLERANCE = 1e-6                              # numbers closer than this count as equal


def flatten(data, prefix=""):
    """Nested dict -> {"a / b / c": value} so every setting is one line."""
    flat = {}
    for key, value in data.items():
        full_key = f"{prefix} / {key}" if prefix else str(key)
        if isinstance(value, dict):
            flat.update(flatten(value, full_key))
        else:
            flat[full_key] = value
    return flat


def same(a, b):
    """Equal values, with a tolerance for numbers (also inside lists)."""
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        if a == b:                               # also covers inf == inf
            return True
        return abs(a - b) <= TOLERANCE
    if isinstance(a, list) and isinstance(b, list) and len(a) == len(b):
        for x, y in zip(a, b):
            if not same(x, y):
                return False
        return True
    return a == b


with open(FILE_A) as f:
    a = flatten(json.load(f))
with open(FILE_B) as f:
    b = flatten(json.load(f))
print(f"A: {FILE_A}  ({len(a)} settings)")
print(f"B: {FILE_B}  ({len(b)} settings)")

only_a = sorted(set(a) - set(b))
only_b = sorted(set(b) - set(a))
different = []
for key in sorted(set(a) & set(b)):
    if not same(a[key], b[key]):
        different.append(key)

print(f"\n=== only in A ({len(only_a)}) ===")
for key in only_a:
    print(f"  {key} = {a[key]}")
print(f"\n=== only in B ({len(only_b)}) ===")
for key in only_b:
    print(f"  {key} = {b[key]}")
print(f"\n=== different ({len(different)}) ===")
for key in different:
    value_a = a[key]
    value_b = b[key]
    if isinstance(value_a, str) and "\n" in value_a:        # e.g. the script node's script
        value_a = "<text, see file>"
        value_b = "<text, see file>"
    print(f"  {key}\n      A: {value_a}\n      B: {value_b}")

if not (only_a or only_b or different):
    print("\nNo differences: both scenes use the same settings.")

A: /home/tommy/Documents/autonomous-systems/athena/athena-data-analysis/sim-to-real-gap/stage-2-testing/compare-stage1-stage2/scene1_settings.json  (1222 settings)
B: /home/tommy/Documents/autonomous-systems/athena/athena-data-analysis/sim-to-real-gap/stage-2-testing/compare-stage1-stage2/scene1_settings-new.json  (1222 settings)

=== only in A (0) ===

=== only in B (0) ===

=== different (2) ===
  graphs / /Graph_athena/ackermann_athena / script_node / inputs / inputs:script
      A: <text, see file>
      B: <text, see file>
  graphs / /Graph_athena/ackermann_athena / script_node / inputs / inputs:scriptPath
      A: 
      B: ../scripts/steering_script_node_inline.py
